# radar-imu-fusion: analysis notebook

Interactive walkthrough of the ESKF sensor-fusion pipeline: generate ground truth + sensor data once, then inspect the raw sensor data, run a couple of scenarios, and look at the internals (RANSAC inlier selection, Doppler Jacobian, NEES) in more detail than the static `results/*.png` files from `python -m src.run_fusion`.

Run from the `radar-imu-fusion/` directory (or adjust `sys.path` below) with the `radar-imu-fusion` environment (`pip install -r requirements.txt`).

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root))

import numpy as np
import yaml
import matplotlib.pyplot as plt
%matplotlib inline

from src.trajectory import generate_trajectory
from src.sensors.imu import simulate_imu
from src.sensors.radar import simulate_radar
from src.sensors.camera import simulate_camera
from src.sensors.gnss import simulate_gnss
from src.filters.eskf import ESKF, NominalState, build_initial_covariance
from src.filters.lie_group import inject_rotation_error, log_so3
from src.filters.measurement_models import doppler_ego_velocity_model, ransac_doppler_inliers
from src.utils.rotations import quat_to_rot
from src.run_fusion import run_scenario, SCENARIOS, generate_all_sensor_data

with open(repo_root / "config" / "sim_params.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["seed"]

## 1. Ground truth + sensor data

Generated once and reused for every scenario below, so comparisons are apples-to-apples (same IMU noise realization, same radar clutter/outlier draws).

In [ ]:
traj, imu, radar_frames, landmarks, moving_objects, camera_frames, gnss_full, gnss_dropout = generate_all_sensor_data(cfg)
print(f"trajectory samples: {len(traj.time)} @ {traj.rate_hz} Hz")
print(f"IMU samples:        {len(imu.time)} @ {imu.rate_hz} Hz")
print(f"radar frames:       {len(radar_frames)}, {len(landmarks)} static landmarks, {len(moving_objects)} moving objects")
print(f"camera frames:      {len(camera_frames)}")
print(f"GNSS fixes:         {len(gnss_full.time)} (full), {len(gnss_dropout.time)} (with 40-80s dropout)")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(traj.position[:, 0], traj.position[:, 1], color="k", lw=2, label="ego trajectory")
ax.scatter(landmarks[:, 0], landmarks[:, 1], s=3, alpha=0.4, color="tab:blue", label="static landmarks")
for obj in moving_objects:
    ts = np.linspace(obj.t0, obj.t0 + 8, 20)
    pts = np.array([obj.position(t) for t in ts])
    ax.plot(pts[:, 0], pts[:, 1], color="tab:red", lw=1.5)
ax.set_aspect("equal", adjustable="datalim")
ax.set_xlabel("East [m]"); ax.set_ylabel("North [m]")
ax.legend(); ax.set_title("Simulated world: ego path, static landmarks, moving objects (red)")
plt.show()

## 2. RANSAC Doppler ego-velocity, up close

Pick one radar frame that has a mix of static, moving-object, and clutter detections and look at what RANSAC keeps.

In [ ]:
rng = np.random.default_rng(0)
R_br = np.array(cfg["radar"]["R_body_to_radar"], dtype=float)

example_frame = None
for f in radar_frames:
    n_dyn = sum(1 for d in f.detections if d.is_dynamic)
    n_clutter = sum(1 for d in f.detections if d.is_clutter)
    if n_dyn >= 2 and n_clutter >= 2 and len(f.detections) > 20:
        example_frame = f
        break

idx = traj.index_at(example_frame.time)
state = NominalState(p=traj.position[idx], v=traj.velocity[idx], q=traj.quaternion[idx], ba=np.zeros(3), bg=np.zeros(3))
result = doppler_ego_velocity_model(
    state, example_frame.detections, R_br, cfg["radar"]["sigma_doppler_mps"],
    cfg["eskf"]["ransac_iterations"], cfg["eskf"]["ransac_inlier_threshold_mps"],
    cfg["eskf"]["ransac_min_inliers"], rng,
)
H, innovation, R_meas, inliers = result
inlier_set = set(inliers.tolist())

print(f"frame t={example_frame.time:.2f}s: {len(example_frame.detections)} detections -> {len(inliers)} RANSAC inliers")
for i, d in enumerate(example_frame.detections):
    tag = "INLIER " if i in inlier_set else "outlier"
    truth = "dynamic" if d.is_dynamic else ("clutter" if d.is_clutter else "static ")
    if i < 15:
        print(f"  [{tag}] doppler={d.doppler:+6.2f} m/s  ground-truth={truth}")

RANSAC has no access to the `is_dynamic`/`is_clutter` ground-truth labels above -- it only sees `(azimuth, elevation, doppler)`. It should still reject essentially all dynamic/clutter detections and keep the static ones, purely from consensus on the ego-velocity model.

In [ ]:
n_dyn_total = sum(1 for d in example_frame.detections if d.is_dynamic)
n_clutter_total = sum(1 for d in example_frame.detections if d.is_clutter)
n_dyn_kept = sum(1 for i, d in enumerate(example_frame.detections) if d.is_dynamic and i in inlier_set)
n_clutter_kept = sum(1 for i, d in enumerate(example_frame.detections) if d.is_clutter and i in inlier_set)
print(f"dynamic detections kept as inliers:  {n_dyn_kept} / {n_dyn_total}")
print(f"clutter detections kept as inliers:  {n_clutter_kept} / {n_clutter_total}")
print(f"innovation on kept inliers: mean={innovation.mean():+.4f} m/s, std={innovation.std():.4f} m/s (sigma_d={cfg['radar']['sigma_doppler_mps']})")

## 3. Run two scenarios and compare

`radar_doppler` (Doppler ego-velocity only, no position fixes) vs `radar_full` (+ known-landmark polar position updates). See the main README for the full 6-scenario comparison and discussion of *why* Doppler alone can't hold heading.

In [ ]:
hist_doppler = run_scenario("radar_doppler", traj, imu, radar_frames, landmarks, camera_frames, gnss_full, cfg, seed=cfg["seed"])
hist_full = run_scenario("radar_full", traj, imu, radar_frames, landmarks, camera_frames, gnss_full, cfg, seed=cfg["seed"])

for name, hist in [("radar_doppler", hist_doppler), ("radar_full", hist_full)]:
    err = np.linalg.norm(hist.p_est - hist.p_true, axis=1)
    print(f"{name:15s}: final={err[-1]:8.2f} m  mean={err.mean():8.2f} m  max={err.max():8.2f} m")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for name, hist, color in [("radar_doppler", hist_doppler, "tab:green"), ("radar_full", hist_full, "tab:cyan")]:
    err = np.linalg.norm(hist.p_est - hist.p_true, axis=1)
    ax.plot(hist.time, err, color=color, label=name)
ax.set_yscale("log")
ax.set_xlabel("time [s]"); ax.set_ylabel("position error [m] (log scale)")
ax.legend(); ax.grid(alpha=0.3)
ax.set_title("Adding landmark position fixes breaks the Doppler-only heading ambiguity")
plt.show()

## 4. Filter consistency (NEES)

See `src/utils/plotting.py::plot_nees` for the version used in `results/`; here's the raw computation inline for `radar_full`, which (unlike `radar_doppler`-only) is fully observable and should be roughly consistent with the chi-squared bounds for a 15-dof error state.

In [ ]:
from scipy.stats import chi2
dof = 15
lower, upper = chi2.ppf(0.025, dof), chi2.ppf(0.975, dof)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(hist_full.time, hist_full.nees, color="tab:cyan", lw=0.8)
ax.axhline(dof, color="k", lw=1)
ax.axhline(lower, color="tab:red", ls="--", lw=1)
ax.axhline(upper, color="tab:red", ls="--", lw=1)
ax.set_yscale("log")
ax.set_xlabel("time [s]"); ax.set_ylabel("NEES")
ax.set_title("radar_full: NEES vs 95% chi-squared(15) bounds")
plt.show()

## Next steps

- Run `python -m src.run_fusion --scenario all` from the repo root to regenerate the full `results/` plot set (all 6 scenarios, 9 plot types).
- See the README for the complete results table, the polar-vs-Cartesian noise discussion, and the Doppler-only observability finding.